# FotMob Bundesliga Matchday Extractor

This notebook retrieves all Bundesliga fixtures for a matchday selected with `input()`. It opens the FotMob Bundesliga fixtures page with Selenium and `undetected_chromedriver`, extracts one match ID, and uses that match as the entry point to FotMob's league API for the complete matchday.

FotMob's fixtures-page `round` parameter is zero-based relative to the Bundesliga matchday: matchday 1 uses `round=0`, matchday 2 uses `round=1`, and so on. The notebook therefore calculates `fotmob_round = matchday - 1` and separately validates that the API round equals the requested matchday.

The primary output is `match_ids_{matchday}_fotmob.json` in its configured project output directory. It contains a top-level list whose entries have exactly these fields, in this order: `home_team`, `home_team_id`, `away_team`, `away_team_id`, and `match_id`. A Pandas DataFrame displays the same data for inspection.


In [1]:
# Resolve the project root and import authoritative data locations.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    FOTMOB_MATCH_IDS_DIR,
    ensure_directory,
)


## Imports, constants, and errors

Import the required libraries, define the Bundesliga identifiers and output columns, and provide one notebook-specific exception type for clear failures.


In [2]:
# Import the libraries required by this notebook step.
import json
from typing import Any
from urllib.parse import urlsplit

import pandas as pd
import undetected_chromedriver as uc
from IPython.display import display
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait


league_id = 54
league_slug = "bundesliga"
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 30
# Set workflow configuration value: OUTPUT_COLUMNS.
OUTPUT_COLUMNS = [
    "home_team",
    "home_team_id",
    "away_team",
    "away_team_id",
    "match_id",
]


# Define Fot Mob Extraction Error to keep related behaviour explicit.
class FotMobExtractionError(RuntimeError):
    """Raised when FotMob fixture data cannot be retrieved or validated."""


## Select the Bundesliga matchday

Prompt for the actual Bundesliga matchday, require an integer of at least 1, and derive FotMob's zero-based fixtures-page round parameter.


In [3]:
# Handle for matchday for reuse in the workflow.
def prompt_for_matchday() -> int:
    """Read and validate the requested Bundesliga matchday."""
    raw_value = input("Enter Bundesliga matchday: ").strip()
    # Handle expected failures with a clear, actionable message.
    try:
        selected_matchday = int(raw_value)
    except ValueError as exc:
        raise ValueError(
            "Bundesliga matchday must be entered as a whole number."
        ) from exc

    # Validate the input before continuing with later processing.
    if selected_matchday < 1:
        raise ValueError("Bundesliga matchday must be at least 1.")
    return selected_matchday


## Parse and validate FotMob data

These helpers validate fixture links, numeric identifiers, JSON structure, round consistency, team objects, and names before any output is written.


In [4]:
# Extract first match ID for reuse in the workflow.
def extract_first_match_id(fixture_elements: list[Any]) -> int:
    """Return the first valid numeric match ID from fixture links."""
    # Validate the input before continuing with later processing.
    if not fixture_elements:
        raise FotMobExtractionError(
            "No elements with data-testid='livescores-match' were found."
        )

    invalid_reasons: list[str] = []
    # Process each available item while preserving the current workflow state.
    for position, element in enumerate(fixture_elements, start=1):
        href = element.get_attribute("href")
        if not href:
            invalid_reasons.append(f"fixture {position} has no href")
            continue

        fragment = urlsplit(href).fragment.strip()
        if not fragment:
            invalid_reasons.append(f"fixture {position} href contains no # match ID")
            continue
        if not fragment.isdigit():
            invalid_reasons.append(
                f"fixture {position} has a non-numeric match ID after #"
            )
            continue

        match_id = int(fragment)
        if match_id < 1:
            invalid_reasons.append(f"fixture {position} has an invalid match ID")
            continue
        return match_id

    details = "; ".join(invalid_reasons)
    raise FotMobExtractionError(
        "Fixture links were found, but none contained a valid numeric match ID"
        + (f": {details}." if details else ".")
    )


# Parse and validate api response for reuse in the workflow.
def parse_api_response(response_text: str) -> dict[str, Any]:
    """Parse the API body and require a top-level JSON object."""
    # Handle expected failures with a clear, actionable message.
    try:
        data = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise FotMobExtractionError(
            "FotMob returned malformed JSON "
            f"(line {exc.lineno}, column {exc.colno})."
        ) from exc

    # Validate the input before continuing with later processing.
    if not isinstance(data, dict):
        raise FotMobExtractionError(
            "The FotMob API response must be a top-level JSON object."
        )
    return data


# Validate matches in round for reuse in the workflow.
def validate_matches_in_round(
    data: dict[str, Any], matchday: int
) -> list[dict[str, Any]]:
    """Validate the matches list and every available API round value."""
    # Validate the input before continuing with later processing.
    if "matchesInRound" not in data:
        raise FotMobExtractionError(
            "The FotMob API response has no 'matchesInRound' field."
        )

    matches = data["matchesInRound"]
    # Validate the input before continuing with later processing.
    if not isinstance(matches, list):
        raise FotMobExtractionError(
            "The FotMob 'matchesInRound' field must be a list."
        )
    # Validate the input before continuing with later processing.
    if not matches:
        raise FotMobExtractionError(
            "The FotMob 'matchesInRound' list is empty."
        )

    expected_round = str(matchday)
    # Validate the input before continuing with later processing.
    if data.get("currentRound") is not None:
        current_round = str(data["currentRound"]).strip()
        # Validate the input before continuing with later processing.
        if current_round != expected_round:
            raise FotMobExtractionError(
                f"FotMob API currentRound is {current_round!r}, but requested "
                f"Bundesliga matchday {expected_round}."
            )

    validated_matches: list[dict[str, Any]] = []
    # Process each available item while preserving the current workflow state.
    for position, match in enumerate(matches, start=1):
        # Validate the input before continuing with later processing.
        if not isinstance(match, dict):
            raise FotMobExtractionError(
                f"Match {position} in 'matchesInRound' is not a JSON object."
            )

        # Validate the input before continuing with later processing.
        if match.get("roundName") is not None:
            round_name = str(match["roundName"]).strip()
            # Validate the input before continuing with later processing.
            if round_name != expected_round:
                raise FotMobExtractionError(
                    f"Match {position} has roundName {round_name!r}, but requested "
                    f"Bundesliga matchday {expected_round}."
                )
        validated_matches.append(match)

    return validated_matches


# Handle positive integer for reuse in the workflow.
def require_positive_integer(value: Any, field_name: str) -> int:
    """Convert an integer or digit string to a positive Python integer."""
    # Validate the input before continuing with later processing.
    if isinstance(value, bool):
        raise FotMobExtractionError(f"{field_name} must be a numeric ID.")

    # Validate the input before continuing with later processing.
    if isinstance(value, int):
        numeric_value = value
    # Validate the input before continuing with later processing.
    elif isinstance(value, str) and value.strip().isdigit():
        numeric_value = int(value.strip())
    else:
        raise FotMobExtractionError(f"{field_name} must be a numeric ID.")

    # Validate the input before continuing with later processing.
    if numeric_value < 1:
        raise FotMobExtractionError(f"{field_name} must be greater than zero.")
    return numeric_value


# Handle team for reuse in the workflow.
def require_team(
    match: dict[str, Any], side: str, position: int
) -> tuple[str, int]:
    """Return a validated team name and ID for one side of a fixture."""
    team = match.get(side)
    # Validate the input before continuing with later processing.
    if not isinstance(team, dict):
        raise FotMobExtractionError(
            f"Match {position} has no valid '{side}' team object."
        )

    name = team.get("name")
    # Validate the input before continuing with later processing.
    if not isinstance(name, str) or not name.strip():
        raise FotMobExtractionError(
            f"Match {position} has no valid {side} team name."
        )

    team_id = require_positive_integer(
        team.get("id"), f"Match {position} {side} team ID"
    )
    return name.strip(), team_id


# Build output data for reuse in the workflow.
def build_output_data(
    matches: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Extract only the required output fields while preserving API order."""
    output_data: list[dict[str, Any]] = []
    # Process each available item while preserving the current workflow state.
    for position, match in enumerate(matches, start=1):
        home_team, home_team_id = require_team(match, "home", position)
        away_team, away_team_id = require_team(match, "away", position)
        match_id = require_positive_integer(
            match.get("id"), f"Match {position} match ID"
        )
        output_data.append(
            {
                "home_team": home_team,
                "home_team_id": home_team_id,
                "away_team": away_team,
                "away_team_id": away_team_id,
                "match_id": match_id,
            }
        )
    return output_data


## Retrieve the fixture entry point and complete matchday

Open the fixtures page, wait explicitly for fixture links, extract one valid match ID, and load the league API response in the same browser session. The `finally` block closes Chrome even when retrieval or parsing fails.


In [5]:
# Handle matchday data for reuse in the workflow.
def retrieve_matchday_data(
    fixtures_url: str, selected_league_id: int
) -> tuple[dict[str, Any], int]:
    """Retrieve a complete FotMob matchday using one Chrome session."""
    driver = None
    # Handle expected failures with a clear, actionable message.
    try:
        # Handle expected failures with a clear, actionable message.
        try:
            driver = uc.Chrome(version_main = 150)
            driver.set_page_load_timeout(WAIT_TIMEOUT_SECONDS)
        except Exception as exc:
            raise FotMobExtractionError(
                f"Could not start undetected Chrome: {exc}"
            ) from exc

        # Handle expected failures with a clear, actionable message.
        try:
            driver.get(fixtures_url)
            fixture_elements = WebDriverWait(
                driver, WAIT_TIMEOUT_SECONDS
            ).until(
                lambda current_driver: (
                    elements
                    if (
                        elements := current_driver.find_elements(
                            By.CSS_SELECTOR,
                            'a[data-testid="livescores-match"]',
                        )
                    )
                    else False
                )
            )
        except TimeoutException as exc:
            raise FotMobExtractionError(
                "Timed out waiting for livescores-match fixture links on "
                f"{fixtures_url}."
            ) from exc
        except WebDriverException as exc:
            raise FotMobExtractionError(
                f"Could not load the FotMob fixtures page: {exc}"
            ) from exc

        match_id = extract_first_match_id(fixture_elements)
        api_url = (
            "https://www.fotmob.com/api/data/leagueDataForMatch"
            f"?matchId={match_id}&leagueId={selected_league_id}"
        )

        # Handle body text for reuse in the workflow.
        def nonempty_body_text(current_driver: Any) -> str | bool:
            body = current_driver.find_element(By.TAG_NAME, "body")
            body_text = body.text.strip()
            return body_text if body_text else False

        # Handle expected failures with a clear, actionable message.
        try:
            driver.get(api_url)
            response_text = WebDriverWait(
                driver, WAIT_TIMEOUT_SECONDS
            ).until(nonempty_body_text)
        except TimeoutException as exc:
            raise FotMobExtractionError(
                "Timed out waiting for a non-empty FotMob API response."
            ) from exc
        except WebDriverException as exc:
            raise FotMobExtractionError(
                f"Could not load the FotMob league API: {exc}"
            ) from exc

        return parse_api_response(response_text), match_id
    finally:
        if driver is not None:
            # Handle expected failures with a clear, actionable message.
            try:
                driver.quit()
            except Exception as cleanup_error:
                print(
                    "Warning: Chrome could not be closed cleanly: "
                    f"{cleanup_error}"
                )


## Save and inspect the validated fixtures

Write only the required fields to a relative UTF-8 JSON file, preserving Unicode and FotMob's match order. Display the same rows with an explicit DataFrame column order.


In [6]:
# Save output data for reuse in the workflow.
def save_output_data(
    output_data: list[dict[str, Any]], output_path: Path
) -> None:
    """Write match data as indented UTF-8 JSON in the runtime directory."""
    # Handle expected failures with a clear, actionable message.
    try:
        # Use the resource only within this controlled scope.
        with output_path.open("w", encoding="utf-8") as output_file:
            json.dump(
                output_data,
                output_file,
                ensure_ascii=False,
                indent=2,
            )
            output_file.write("\n")
    except OSError as exc:
        raise FotMobExtractionError(
            f"Could not write output file {output_path!r}: {exc}"
        ) from exc


## Run the extraction

Prompt for the matchday, retrieve and validate the API data, save the JSON, display the DataFrame, and print a concise execution summary.


In [7]:
# Run this self-contained workflow step using the prepared inputs.
matchday = prompt_for_matchday()
fotmob_round = matchday - 1
fixtures_url = (
    f"https://www.fotmob.com/leagues/{league_id}/fixtures/"
    f"{league_slug}?group=by-round&round={fotmob_round}"
)

api_data, entry_match_id = retrieve_matchday_data(fixtures_url, league_id)
matches_in_round = validate_matches_in_round(api_data, matchday)
output_data = build_output_data(matches_in_round)

if len(output_data) != 9:
    print(
        "Warning: Expected approximately 9 Bundesliga fixtures but retrieved "
        f"{len(output_data)}."
    )

output_path = ensure_directory(FOTMOB_MATCH_IDS_DIR) / f"match_ids_{matchday}_fotmob.json"
save_output_data(output_data, output_path)

df = pd.DataFrame(output_data, columns=OUTPUT_COLUMNS)
display(df)

api_matchday = (
    str(api_data["currentRound"]).strip()
    if api_data.get("currentRound") is not None
    else str(matchday)
)
print(f"Bundesliga matchday: {matchday}")
print(f"FotMob fixtures round parameter: {fotmob_round}")
print(f"API matchday: {api_matchday}")
print(f"Matches retrieved: {len(output_data)}")
print(f"Output saved to: {output_path}")


Enter Bundesliga matchday:  1


,home_team,home_team_id,away_team,away_team_id,match_id
0,Bayern München,9823,VfB Stuttgart,10269,5881143
1,Elversberg,8232,Bayer Leverkusen,8178,5881146
2,1. FC Köln,8722,Hoffenheim,8226,5881147
3,Mainz 05,9905,Paderborn,8460,5881149
4,RB Leipzig,178475,Borussia Mönchengladbach,9788,5881150
5,Union Berlin,8149,Eintracht Frankfurt,9810,5881151
6,Borussia Dortmund,9789,Hamburger SV,9790,5881145
7,Freiburg,8358,Werder Bremen,8697,5881148
8,Augsburg,8406,Schalke 04,10189,5881144


Bundesliga matchday: 1
FotMob fixtures round parameter: 0
API matchday: 1
Matches retrieved: 9
Output saved to: C:\kickbase project\outputs\fotmob\match_ids\match_ids_1_fotmob.json
